# 4. Detection Algorithms

All detectors share one constructor, `peyes.create_detector(algorithm, missing_value, min_event_duration,
pad_blinks_time, **kwargs)`, and one method, `.detect(t, x, y, viewer_distance_cm, pixel_size_cm)`. What differs
between algorithms is the `**kwargs` and the set of event labels each one can produce. This notebook covers two
algorithms in depth — IVT (simplest) and Engbert (used elsewhere in this guide) — then a quick tour of the rest.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()

## IVT — velocity threshold

I-VT (Salvucci & Goldberg, 2000) is the simplest algorithm: samples above a velocity threshold are saccades,
everything else is a fixation. Its only parameter is that threshold, `saccade_velocity_threshold` (deg/s,
default 45). Every detector's `.documentation()` prints its algorithm description, parameters, and citation:

In [2]:
ivt = peyes.create_detector(
    "ivt", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0, saccade_velocity_threshold=45,
)
print(ivt.documentation())

Algorithm:	IVT
Articles:
- Salvucci, D. D., & Goldberg, J. H. (2000). Identifying fixations and saccades in eye-tracking protocols. In Proceedings of the Symposium on Eye Tracking Research & Applications (pp. 71-78)

Implements the I-VT (velocity threshold) gaze event detection algorithm, as described in:
    Salvucci, D. D., & Goldberg, J. H. (2000). Identifying fixations and saccades in eye-tracking protocols.
    In Proceedings of the Symposium on Eye Tracking Research & Applications (pp. 71-78).

General algorithm:
1. Calculate the angular velocity of the gaze data
2. Identify saccade candidates as samples with angular velocity greater than the threshold
3. Assume undefined (non-blink) samples are fixations

:param missing_value: the value that indicates missing data in the gaze data.
:param min_event_duration: the minimum duration of a gaze event, in milliseconds.
:param pad_blinks_ms: the duration to pad around detected blinks, in milliseconds.
:param name: the name of the detect

In [3]:
labels, _ = ivt.detect(t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"])
events = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
peyes.summarize_events(events)["event_type"].value_counts()

event_type
FIXATION    25
SACCADE     25
BLINK        1
Name: count, dtype: int64

Lowering the threshold makes the detector more sensitive to small velocity changes, splitting off more
(shorter) saccades:

In [4]:
ivt_sensitive = peyes.create_detector(
    "ivt", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0, saccade_velocity_threshold=20,
)
labels_sensitive, _ = ivt_sensitive.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
events_sensitive = peyes.create_events(
    labels=labels_sensitive, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
peyes.summarize_events(events_sensitive)["event_type"].value_counts()

event_type
FIXATION    64
SACCADE     64
BLINK        1
Name: count, dtype: int64

## Engbert — median-based velocity ellipse

The Engbert & Kliegl (2003) algorithm (used in notebooks 1-3) also separates fixations from saccades by velocity,
but the threshold is data-driven: an elliptical threshold scaled by the median-based standard deviation of the
velocity signal, controlled by `lambda_param` (default 5) and the derivative window `deriv_window_size`
(default 5 samples):

In [5]:
engbert = peyes.create_detector(
    "engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0, lambda_param=5, deriv_window_size=5,
)
labels, _ = engbert.detect(t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"])
events = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
peyes.summarize_events(events)["event_type"].value_counts()

event_type
SACCADE     31
FIXATION    30
BLINK        1
Name: count, dtype: int64

## The other five algorithms

`create_detector` takes the same three positional arguments for every algorithm; only the `**kwargs` change. Beyond
IVT and Engbert:

| key | algorithm | adds | typical use |
|---|---|---|---|
| `"ivvt"` | I-VVT | a second, lower velocity threshold | separates smooth pursuit from fixations too |
| `"idt"` | I-DT | dispersion threshold over a sliding window | robust to noisy velocity estimates |
| `"idvt"` | I-DVT | dispersion (fixation/pursuit) + velocity (pursuit/saccade) | pursuit-aware, dispersion-based |
| `"nh"` | Nystrom & Holmqvist | adaptive thresholds, PSO detection | literature-standard, more parameters to tune |
| `"remodnav"` | REMoDNaV | extends NH with smooth-pursuit detection | full label set (fixation/saccade/PSO/pursuit) |

Each has its own kwargs — see `create_detector`'s docstring (`help(peyes.create_detector)`) or, once built, a
detector's own `.documentation()`. Running all seven with their defaults on the same trial:

In [6]:
results = {}
for algorithm in ["ivt", "ivvt", "idt", "idvt", "engbert", "nh", "remodnav"]:
    detector = peyes.create_detector(algorithm, missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
    labels, _ = detector.detect(
        t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
    )
    results[algorithm] = labels

import pandas as pd
pd.DataFrame({name: pd.Series(labels).value_counts() for name, labels in results.items()}).fillna(0).astype(int)

C:\Users\nirjo\Documents\University\PhD\Projects\pEYES\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\nirjo\Documents\University\PhD\Projects\pEYES\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,ivt,ivvt,idt,idvt,engbert,nh,remodnav
0,2,51,26,3,13,5,17
1,2406,2124,1568,1520,2285,2289,758
2,329,344,1190,311,439,434,243
3,0,0,0,0,0,56,95
4,0,218,0,903,0,0,1624
5,47,47,0,47,47,0,47


Each row is an `EventLabelEnum` value (0=UNDEFINED, 1=FIXATION, 2=SACCADE, 3=PSO, 4=SMOOTH_PURSUIT,
5=BLINK). Notice only `ivvt`, `idvt`, and `remodnav` ever produce label 4 (smooth pursuit) — the others simply
don't model that event type. A quick visual comparison (full visualization coverage is in notebooks 10-11):

In [7]:
peyes.visualize.scarfplot_comparison_figure(
    d["t"], *results.values(), names=list(results.keys()),
)

## What's next

**[5 Sample-Level Evaluation](./5%20Sample-Level%20Evaluation.ipynb)** — quantifying how much detectors agree with
each other (or with a human rater), directly on these label arrays.